![dvd_image](dvd_image.jpg)

A DVD rental company needs your help! They want to figure out how many days a customer will rent a DVD for based on some features and has approached you for help. They want you to try out some regression models which will help predict the number of days a customer will rent a DVD for. The company wants a model which yeilds a MSE of 3 or less on a test set. The model you make will help the company become more efficient inventory planning.

The data they provided is in the csv file `rental_info.csv`. It has the following features:
- `"rental_date"`: The date (and time) the customer rents the DVD.
- `"return_date"`: The date (and time) the customer returns the DVD.
- `"amount"`: The amount paid by the customer for renting the DVD.
- `"amount_2"`: The square of `"amount"`.
- `"rental_rate"`: The rate at which the DVD is rented for.
- `"rental_rate_2"`: The square of `"rental_rate"`.
- `"release_year"`: The year the movie being rented was released.
- `"length"`: Lenght of the movie being rented, in minuites.
- `"length_2"`: The square of `"length"`.
- `"replacement_cost"`: The amount it will cost the company to replace the DVD.
- `"special_features"`: Any special features, for example trailers/deleted scenes that the DVD also has.
- `"NC-17"`, `"PG"`, `"PG-13"`, `"R"`: These columns are dummy variables of the rating of the movie. It takes the value 1 if the move is rated as the column name and 0 otherwise. For your convinience, the reference dummy has already been dropped.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error as MSE

# Import any additional modules and start coding below
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

# convert to datetime
df = pd.read_csv("rental_info.csv")

df["return_date"] = pd.to_datetime(df["return_date"])
df["rental_date"] = pd.to_datetime(df["rental_date"])
# get number of rental days
df["rental_length"] = df["return_date"] - df["rental_date"]
df["rental_length_days"] = df["rental_length"].dt.days

# dummy variables
df["deleted_scenes"] = np.where(df["special_features"].str.contains("Deleted Scenes"), 1, 0)
df["behind_the_scenes"] = np.where(df["special_features"].str.contains("Behind the Scenes"), 1, 0)

# train test split
y = df["rental_length_days"]
X = df.drop(columns=["rental_length_days", "rental_date", "return_date", "special_features","rental_length"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=9)

# feature selection
lasso = Lasso(alpha=0.3, random_state = 9)
lasso.fit(X_train, y_train)
lasso_coef = lasso.coef_

X_lasso_train, X_lasso_test = X_train.iloc[:, lasso_coef > 0], X_test.iloc[:, lasso_coef > 0]

# fitting and tuning the model
# random forest
rf = RandomForestRegressor()
param_dist = {
    'n_estimators' : np.arange(1, 101, 1),
    'max_depth' : np.arange(1, 11, 1),
    'min_samples_leaf' : [1, 2, 4],
    'min_samples_split' : [2, 5, 10],
    'max_features' : ['sqrt', 'log2']
}

rf_random = RandomizedSearchCV(
    estimator = rf,
    param_distributions = param_dist,
    cv = 5,
    random_state = 9
)
rf_random.fit(X_train, y_train)

best_model = rf_random.best_estimator_

y_pred = best_model.predict(X_test)
best_mse = MSE(y_test, y_pred)

print("Best Model: ", best_model)
print("RMSE: ", best_mse)


Best Model:  RandomForestRegressor(max_depth=10, max_features='sqrt', n_estimators=57)
RMSE:  2.2599372029950504
